# CALI-PRED — fast Colab run

Runtime → Change runtime type → **T4 GPU** before you start.

Run the cells in order. Cell 5 is a two-minute smoke test — **do not skip it**; it catches a broken
patch before you spend an hour on the full run.

### Why the old run took hours

`ImputationReliabilityEngine.compute_ensemble()` runs once per window, and each call built two fresh
models, copied them to the GPU, created two Adam optimizers, and trained each for 30 steps on a
`(1, 60, 15)` tensor. At stride 100 that is 15,169 windows per seed:

* 910,140 forward+backward passes on 900-element tensors
* 30,338 module constructions and host-to-device copies

A T4 cannot help with that. The tensors are far too small to occupy the device, so the wall clock is
kernel-launch latency and Python overhead, and the GPU idles. `colab_speedups.py` builds the models
once, moves the ensemble to CPU where there is no launch overhead to amortize, and caches the
precompute so seeds 2 and 3 reuse seed 1's work.

## 1. Get the code

In [ ]:
%cd /content
![ -d CALI-PRED ] && (cd CALI-PRED && git pull) || git clone https://github.com/Armin1126/CALI-PRED.git
%cd /content/CALI-PRED
!git log --oneline -3

## 2. Dependencies

In [ ]:
!pip -q install numpy pandas scipy scikit-learn matplotlib
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 3. The dataset

The MetroPT CSV is 218 MB and gitignored, so it is not in the clone. Your repo already has a
downloader that pulls it from UCI — try that first; Colab's connection is fast and it needs no
Drive setup. Mounting Drive as well means you only download once, ever.

**Do not skip the verification at the end of the cell.** `IndustrialDataLoader` used to catch a
missing file, log an error, and then quietly continue on 2,000 timesteps of synthetic data — the
run completes, and the numbers look perfectly normal. Cell 4 patches that to raise instead.

In [ ]:
import os, subprocess
DATA = 'data/metropt/MetroPT3(AirCompressor).csv'
os.makedirs('data/metropt', exist_ok=True)

# Optional but recommended: Drive keeps the CSV across runtime resets.
DRIVE = '/content/drive/MyDrive/CALI-PRED-data'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE, exist_ok=True)
except Exception as e:
    DRIVE = None
    print('Drive not mounted, continuing without it:', e)

cached = f'{DRIVE}/MetroPT3(AirCompressor).csv' if DRIVE else None

if not os.path.exists(DATA):
    if cached and os.path.exists(cached):
        os.symlink(cached, DATA); print('linked from Drive')
    else:
        print('Downloading from UCI...')
        subprocess.run(['python', 'download_data.py', '--dataset', 'metropt'], check=False)
        if os.path.exists(DATA) and cached:
            import shutil; shutil.copy2(DATA, cached)
            print('copied to Drive for next time')

# Verify. A wrong path here is worse than a crash: the loader will
# substitute synthetic data and the run will look completely normal.
assert os.path.exists(DATA), (
    f'{DATA} still missing. Download the MetroPT-3 zip from\n'
    '  https://archive.ics.uci.edu/dataset/791/metropt+3+dataset\n'
    'and upload the CSV to data/metropt/ using the file browser on the left.')

import pandas as pd
n = sum(1 for _ in open(DATA)) - 1
print(f'{DATA}  |  {os.path.getsize(DATA)/1e6:.0f} MB  |  {n:,} rows')
assert n > 1_000_000, f'only {n:,} rows -- the real record has ~1.5M. Wrong file?'
print('dataset verified')

## 4. Apply the patches

Both scripts keep backups (`.prespeed`, `.orig`) and are safe to run twice.

`rerun_for_paper.py` additionally restricts the forecast to the 7 continuous channels and wires
`calibration_weight` to the CLI — see its docstring for why.

In [ ]:
import os
os.environ['CALIPRED_IRI_DEVICE']      = 'cpu'    # tiny tensors: CPU wins
os.environ['CALIPRED_IRI_EPOCHS']      = '30'     # try 15 once a full run is verified
os.environ['CALIPRED_CORRUPTION_SEED'] = '12345'  # pinned so the cache is shared
os.environ['CALIPRED_CACHE_DIR']       = (
    '/content/drive/MyDrive/CALI-PRED-cache'
    if os.path.isdir('/content/drive/MyDrive') else 'precompute_cache')
# Leave CALIPRED_ALLOW_MOCK unset. Setting it to '1' re-enables the silent
# synthetic-data fallback, which is only ever right for a deliberate dry run.

!python colab_speedups.py

## 5. Smoke test — run this before the full job

Two minutes. Confirms the patches did not break anything and gives you a timing estimate.
If it fails, stop and restore with `cp pipeline.py.prespeed pipeline.py` (same for `iri_module.py`).

In [ ]:
import time
t0 = time.time()
!python pipeline.py --data-path "$DATA" --max-windows 200 --epochs 2 \
    --stride 100 --checkpoint-dir checkpoints_smoke
dt = time.time() - t0
print(f'\nsmoke test: {dt:.0f}s for 200 windows')
print(f'extrapolated, 15,169 windows: ~{dt*15169/200/60:.0f} min for the first seed')
print('seeds 2 and 3 skip the precompute entirely (cache hit), so expect them much faster.')

## 6. Full run

Three seeds, unscaled and scaled variants. Drop `--skip-scaled` if you want both.

In [ ]:
!python rerun_for_paper.py --data-path "$DATA" --skip-scaled 2>&1 | tee run_log.txt

## 7. Collect the results

In [ ]:
import json, glob
print(open('paper_numbers.json').read())
print('\nprediction files kept for the paper:')
for p in sorted(glob.glob('checkpoints_paper_*/seed_*/test_predictions.npz')):
    print(' ', p)

In [ ]:
# Keep the predictions this time -- they are what the paper's numbers are
# recomputed from, and a runtime reset destroys anything left in /content.
import shutil, os
dst = '/content/drive/MyDrive/CALI-PRED-results'
if os.path.isdir('/content/drive/MyDrive'):
    os.makedirs(dst, exist_ok=True)
    for d in ['checkpoints_paper_unscaled', 'checkpoints_paper_scaled']:
        if os.path.isdir(d):
            shutil.copytree(d, f'{dst}/{d}', dirs_exist_ok=True)
    for f in ['paper_numbers.json', 'run_log.txt']:
        if os.path.exists(f):
            shutil.copy2(f, dst)
    print('saved to', dst)
else:
    print('Drive not mounted -- download paper_numbers.json and run_log.txt manually.')